In [1]:
import os
import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
conn = psycopg.connect(
    host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)
register_vector(conn)

conn.execute("DROP TABLE IF EXISTS chunks")
conn.execute("""
    CREATE TABLE chunks (
        id SERIAL PRIMARY KEY,
        song_id INT,
        chunk_index INT,
        title TEXT,
        artist TEXT,
        year INT,
        rank INT,
        chunk_text TEXT,
        embedding vector(384)
    )
""")
conn.commit()

In [2]:
chunks_df = pd.read_csv("../data/chunks.csv")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(embeddings.shape)   # expect (18549, 384)

np.save("../data/embeddings.npy", embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/290 [00:00<?, ?it/s]

(18549, 384)


In [3]:
rows = []
for i, row in chunks_df.iterrows():
    rows.append((
        int(row["song_id"]), int(row["chunk_index"]), row["title"], row["artist"],
        int(row["year"]), int(row["rank"]), row["chunk_text"], embeddings[i],
    ))

with conn.cursor() as cur:
    cur.executemany(
        """INSERT INTO chunks
           (song_id, chunk_index, title, artist, year, rank, chunk_text, embedding)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        rows,
    )
conn.commit()

print(conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0])

18549


In [4]:
query = "feeling lonely after a breakup"
q_vec = model.encode(query, normalize_embeddings=True)

results = conn.execute(
    """SELECT title, artist, year, 1 - (embedding <=> %s) AS similarity
       FROM chunks
       ORDER BY embedding <=> %s
       LIMIT 5""",
    (q_vec, q_vec),
).fetchall()

for r in results:
    print(r)

('Even the Nights Are Better', 'air supply', 1982, 0.6459550857543945)
('Back 2 Good', 'matchbox twenty', 1999, 0.5987488031387329)
('U Got It Bad', 'usher', 2002, 0.5602535009384155)
("We Don't Talk Anymore - 2001 Remaster", 'cliff richard', 1980, 0.5568995641738049)
('We Got A Love Thang', 'cece peniston', 1992, 0.5525043771417256)
